# 00 - Setup del Workshop

## Orden de ejecución
1. **Run All** en este notebook (o celda por celda de arriba hacia abajo)
2. Verifique que el catálogo `BNS` aparece en Catalog Explorer
3. Continúe con los labs

> Si `carga_datos` falla por red, suba manualmente `Files/initial/` al volumen.


In [ ]:
# Celda 1 — Sin dependencias externas (carga_datos usa Git folder o GitHub API)
print("Setup BNCR — listo para ejecutar")


In [ ]:
%run ./00_variables


In [ ]:
# Celda 3 — Validar variables antes de crear objetos UC
required = ["catalog_name", "schema_raw", "schema_bronze", "schema_silver", "schema_gold", "volume", "vol_path", "carga_datos"]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"Variables faltantes: {missing}. Ejecute de nuevo la celda %run ./00_variables")
print("Variables OK:", catalog_name, vol_path)


In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"USE CATALOG {catalog_name}")

for s in [schema_raw, schema_bronze, schema_silver, schema_gold]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{s}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_raw}.{volume}")
dbutils.fs.mkdirs(vol_path)

print("Catálogo y esquemas creados:")
display(spark.sql(f"SHOW SCHEMAS IN {catalog_name}"))


In [ ]:
carga_datos("initial")


## Verificación


In [ ]:
display(dbutils.fs.ls(vol_path))


In [ ]:
for folder in dbutils.fs.ls(vol_path):
    if folder.isDir():
        n = len(dbutils.fs.ls(folder.path))
        print(f"{folder.name}: {n} archivos")
